# Astilla — worker headless de render visual (Colab, GPU gratis)

Genera **una imagen estilizada por escena** a partir de `visual_job.json` (exportado por `python -m pipeline.animado`).

- **Checkpoint por escena (ADR-002):** si una imagen ya existe, la salta — reanudable si Colab se desconecta.
- **Determinismo (ADR-006):** la semilla por escena viene en el job.
- Entorno: **Runtime → Cambiar tipo de entorno → GPU (T4)**.

Flujo: 1) sube `visual_job.json`  →  2) corre las celdas  →  3) baja `escenas.zip` y descomprime en `artifacts/escenas/` local  →  4) `python -m pipeline.animado ... --ensamblar`.

In [ ]:
!pip -q install diffusers transformers accelerate

In [ ]:
# Sube el visual_job.json que genero el pipeline local
from google.colab import files
subido = files.upload()  # elige artifacts/visual_job.json

In [ ]:
import json, os, torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

job = json.load(open('visual_job.json', encoding='utf-8'))
g = job['generacion']
os.makedirs('escenas', exist_ok=True)
print('job', job['job_id'], '|', len(job['escenas']), 'escenas | estilo', job['estilo'])

pipe = StableDiffusionPipeline.from_pretrained(
    g['modelo'], torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')

In [ ]:
# Render por escena, con checkpoint (ADR-002): salta las ya hechas
for esc in job['escenas']:
    out = os.path.join('escenas', esc['archivo'])
    if os.path.exists(out):
        print('checkpoint, salto', esc['archivo']); continue
    gen = torch.Generator('cuda').manual_seed(int(esc['semilla']))
    img = pipe(esc['prompt'], negative_prompt=job['negativo'],
               width=g['width'], height=g['height'],
               num_inference_steps=g['steps'], guidance_scale=g['guidance'],
               generator=gen).images[0]
    img.save(out); print('ok', esc['archivo'], '::', esc['prompt'][:60])

In [ ]:
# Empaqueta y descarga las escenas -> descomprime en artifacts/escenas/ local
import zipfile
from google.colab import files
with zipfile.ZipFile('escenas.zip', 'w') as z:
    for f in sorted(os.listdir('escenas')):
        z.write(os.path.join('escenas', f), f)
files.download('escenas.zip')